# Item D

### Carregamento dos dados

In [5]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from medmnist import BloodMNIST
import os

from ml3.config import RANDOM_SEED
torch.manual_seed(RANDOM_SEED)

# escolhido ResNet-18

# Pipeline de Transformações para Transfer Learning (ImageNet)
data_transform = transforms.Compose([
    transforms.Resize((64, 64)), # Redimensiona 28x28 -> 64x64
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

NUM_WORKERS = min(4, os.cpu_count())
print(f"NUM_WORKERS: {NUM_WORKERS}")

# Carregar os datasets oficiais do BloodMNIST (Treino, Validação e Teste)
# O parâmetro download=True baixa a base automaticamente caso não exista localmente
train_dataset = BloodMNIST(split='train', transform=data_transform, download=True)
val_dataset   = BloodMNIST(split='val', transform=data_transform, download=True)
test_dataset  = BloodMNIST(split='test', transform=data_transform, download=True)

# Criar os DataLoaders para gerenciar o mini-batching
BATCH_SIZE = 64

#train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
#val_loader   = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)


# gpu?
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"DEVICE: {DEVICE}")


train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True if DEVICE.type == 'cuda' else False,
    persistent_workers=True if NUM_WORKERS > 0 else False  # Mantém os processos vivos entre épocas
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if DEVICE.type == 'cuda' else False,
    persistent_workers=True if NUM_WORKERS > 0 else False
)

NUM_WORKERS: 4
DEVICE: cuda


### Transfer + Treino

Treino sem freeze

In [6]:
import torch
import torch.nn as nn
import ml3.transfer_d as transfer_d
import pandas as pd

# Carregar ResNet-18 Pré-treinada
model_resnet = transfer_d.get_pretrained_resnet18(num_classes=8, freeze_backbone=False)

total_p, train_p = transfer_d.count_parameters(model_resnet)
print(f"Parâmetros Totais: {total_p:,} | Treináveis: {train_p:,}")

#LR = 1e-4
LR = 1e-4
EPOCHS = 20

# Treinamento
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_resnet.parameters(), lr=LR, weight_decay=1e-4)

print("Iniciando treinamento")

best_model, best_val_acc, train_time, history = transfer_d.train_and_validate_transfer(
    model=model_resnet,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    epochs=EPOCHS,
    device=DEVICE
)

print(f"\nMelhor Acurácia de Validação (ResNet-18): {best_val_acc:.4f}")
print(f"Tempo de Treinamento: {train_time:.2f} segundos")

torch.save(best_model.state_dict(), 'best_model_resnet18_item_d.pth')

df_history = pd.DataFrame(history)
df_history.to_csv("item_d_epochs_history.csv", index=False)

Parâmetros Totais: 11,180,616 | Treináveis: 11,180,616
Iniciando treinamento

Melhor Acurácia de Validação (ResNet-18): 0.9570
Tempo de Treinamento: 188.36 segundos


Treino COM FREEZE

In [7]:
import torch
import torch.nn as nn
import ml3.transfer_d as transfer_d

# Carregar ResNet-18 Pré-treinada
model_resnet = transfer_d.get_pretrained_resnet18(num_classes=8, freeze_backbone=True)

total_p, train_p = transfer_d.count_parameters(model_resnet)
print(f"Parâmetros Totais: {total_p:,} | Treináveis: {train_p:,}")

#LR = 1e-4
LR = 1e-3
EPOCHS = 20

# Treinamento
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_resnet.parameters(), lr=LR, weight_decay=1e-4)

print("Iniciando treinamento")

best_model, best_val_acc, train_time, history = transfer_d.train_and_validate_transfer(
    model=model_resnet,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    epochs=EPOCHS,
    device=DEVICE
)

print(f"\nMelhor Acurácia de Validação (ResNet-18): {best_val_acc:.4f}")
print(f"Tempo de Treinamento: {train_time:.2f} segundos")

torch.save(best_model.state_dict(), 'best_model_freeze_resnet18_item_d.pth')

df_history = pd.DataFrame(history)
df_history.to_csv("item_d_freeze_epochs_history.csv", index=False)

Parâmetros Totais: 11,180,616 | Treináveis: 4,104
Iniciando treinamento

Melhor Acurácia de Validação (ResNet-18): 0.7499
Tempo de Treinamento: 39.16 segundos


### Validação

Sem freeze

In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

df_history = pd.read_csv("item_d_epochs_history.csv")

epochs_x = list(range(1, len(df_history) + 1))
best_ep = int(np.argmax(df_history['val_bal_acc'])) + 1

fig = make_subplots(rows=1, cols=2, subplot_titles=("Loss por época", "Acurácia balanceada (validação)"))
fig.add_trace(go.Scatter(x=epochs_x, y=df_history['train_loss'], name="Treino"), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs_x, y=df_history['val_loss'], name="Validação"), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs_x, y=df_history['val_bal_acc'], name="Acc. bal. (val)", showlegend=False), row=1, col=2)
fig.add_vline(x=best_ep, line_dash="dash", line_color="gray", annotation_text=f"melhor época: {best_ep}")
fig.update_xaxes(title_text="Época")
fig.update_layout(template="plotly_white", width=1000, height=400)
fig.show()

COM FREEZE

In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

df_history = pd.read_csv("item_d_freeze_epochs_history.csv")

epochs_x = list(range(1, len(df_history) + 1))
best_ep = int(np.argmax(df_history['val_bal_acc'])) + 1

fig = make_subplots(rows=1, cols=2, subplot_titles=("Loss por época", "Acurácia balanceada (validação)"))
fig.add_trace(go.Scatter(x=epochs_x, y=df_history['train_loss'], name="Treino"), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs_x, y=df_history['val_loss'], name="Validação"), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs_x, y=df_history['val_bal_acc'], name="Acc. bal. (val)", showlegend=False), row=1, col=2)
fig.add_vline(x=best_ep, line_dash="dash", line_color="gray", annotation_text=f"melhor época: {best_ep}")
fig.update_xaxes(title_text="Época")
fig.update_layout(template="plotly_white", width=1000, height=400)
fig.show()

### Teste

In [10]:
import torch
import plotly.express as px
from sklearn.metrics import classification_report
import ml3.transfer_d as transfer_d

# Nomes das 8 classes do BloodMNIST
class_names = [
    'Basófilo', 'Eosinófilo', 'Eritroblasto', 'Granulócito Imaturo',
    'Linfócito', 'Monócito', 'Neutrófilo', 'Plaqueta'
]

best_resnet = transfer_d.get_pretrained_resnet18(num_classes=8)
best_resnet.load_state_dict(torch.load('best_model_resnet18_item_d.pth'))

res_transfer = transfer_d.evaluate_model(best_resnet, test_loader, device=DEVICE)

print(f"Acurácia Balanceada (Teste) : {res_transfer['bal_acc']:.4f}")
print(f"F1-Score Macro (Teste)      : {res_transfer['f1_macro']:.4f}")

print("--- Relatório Detalhado por Classe ---")
print(classification_report(res_transfer['targets'], res_transfer['preds'], target_names=class_names, digits=4))

# 4. Plotar Matriz de Confusão Interativa (Plotly)
fig_cm = px.imshow(
    res_transfer['cm'],
    x=class_names,
    y=class_names,
    text_auto=True,
    color_continuous_scale='Blues',
    title="<b>Matriz de Confusão - Transfer Learning (ResNet-18, Teste)</b>",
    labels=dict(x="Classe Predita", y="Classe Real", color="Frequência")
)

fig_cm.update_layout(template="plotly_white", width=700, height=600)
fig_cm.show()

Acurácia Balanceada (Teste) : 0.9456
F1-Score Macro (Teste)      : 0.9505
--- Relatório Detalhado por Classe ---
                     precision    recall  f1-score   support

           Basófilo     0.9590    0.9590    0.9590       244
         Eosinófilo     0.9968    0.9920    0.9944       624
       Eritroblasto     0.9641    0.9486    0.9562       311
Granulócito Imaturo     0.8494    0.9447    0.8945       579
          Linfócito     0.9783    0.9259    0.9514       243
           Monócito     0.9261    0.8380    0.8799       284
         Neutrófilo     0.9830    0.9565    0.9696       666
           Plaqueta     0.9979    1.0000    0.9989       470

           accuracy                         0.9544      3421
          macro avg     0.9568    0.9456    0.9505      3421
       weighted avg     0.9565    0.9544    0.9547      3421

